# Why Does CMM Improve Momentum?

This notebook runs paper-style diagnostic tests to explain why characteristic-managed/deep-learning weighted momentum (`CMM`) improves over standard momentum.

Core idea from the paper: all past return days are not equally informative. The neural network maps firm characteristics into a scalar `z_hat`, then uses `softmax(z_hat * daily_return)` to decide which days inside the past-return window receive more weight. The tests below check whether the improvement comes from stronger out-of-sample prediction, incremental information beyond standard momentum, risk exposure, and learned weight concentration.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

SRC_DIR = Path('../src').resolve()
CODE_DIR = Path('../../code').resolve()
for path in [CODE_DIR, SRC_DIR]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
from backtest import neutralize_by_size_industry

DATA_PATH = Path('../result/datasets/cmm_model_training_data.parquet')
PRED_PATH = Path('../result/models/cmm/cmm_predictions.parquet')
RETURN_COLS_PATH = Path('../result/datasets/cmm_return_window_columns.txt')
FEATURE_COLS_PATH = Path('../result/datasets/cmm_feature_columns.txt')
OUT_DIR = Path('../result/reports/cmm_explain')
OUT_DIR.mkdir(parents=True, exist_ok=True)

EVAL_SPLIT = 'test'

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 160)
plt.style.use('seaborn-v0_8-whitegrid')


Matplotlib is building the font cache; this may take a moment.


## 1. Load Out-of-Sample Predictions and Build Baselines

`Standard Momentum` here uses the same CMM return window `ret_lag_252` through `ret_lag_22`, so this section isolates the contribution of learned weighting inside the same historical-return window. The production comparison notebook uses the same paper-style skip-one-month traditional momentum benchmark.

In [2]:
return_cols = RETURN_COLS_PATH.read_text(encoding='utf-8').splitlines()
feature_cols = FEATURE_COLS_PATH.read_text(encoding='utf-8').splitlines()

pred = pd.read_parquet(PRED_PATH)
pred['signal_date'] = pd.to_datetime(pred['signal_date'])
pred = pred[pred['split'] == EVAL_SPLIT].copy()
if pred.duplicated(['stock_id', 'signal_date']).any():
    raise ValueError('Duplicate test predictions detected.')

base_cols = list(dict.fromkeys(['stock_id', 'signal_date', 'target_1m_ret', 'target_1m_ret_cs_z', 'ind1', 'z_nmv'] + return_cols + feature_cols))
base = pd.read_parquet(DATA_PATH, columns=base_cols)
base['signal_date'] = pd.to_datetime(base['signal_date'])
base['ind1'] = base['ind1'].fillna('Unknown')

df = pred[['stock_id', 'signal_date', 'target_1m_ret', 'target_1m_ret_cs_z', 'cmm_signal', 'cmm_signal_cs_z', 'z_hat', 'max_weight', 'fold']].merge(
    base, on=['stock_id', 'signal_date', 'target_1m_ret', 'target_1m_ret_cs_z'], how='inner'
)
df['cmm_neutral'] = neutralize_by_size_industry(df, 'cmm_signal_cs_z')

# Same-window standard momentum: equal-weighted log-return sum over the CMM formation window.
df['same_window_mom'] = df[return_cols].sum(axis=1)
df['same_window_mom_cs_z'] = df.groupby('signal_date')['same_window_mom'].transform(lambda s: (s - s.mean()) / s.std(ddof=0))

print(df.shape)
print(df['signal_date'].min(), df['signal_date'].max(), df['signal_date'].nunique())
df[['stock_id','signal_date','cmm_neutral','same_window_mom_cs_z','z_hat','max_weight','target_1m_ret']].head()


(367699, 331)
2019-01-31 00:00:00 2026-04-30 00:00:00 88


,stock_id,signal_date,cmm_neutral,same_window_mom_cs_z,z_hat,max_weight,target_1m_ret
0,000001.SZ,2019-01-31,-0.413259,0.046867,-2.179826,0.005132,0.139286
1,000002.SZ,2019-01-31,0.769428,-0.209104,1.152991,0.004792,0.031069
2,000004.SZ,2019-01-31,0.786758,0.403541,3.219325,0.005492,0.233267
3,000005.SZ,2019-01-31,0.010900,-0.155188,0.703044,0.004637,0.148148
4,000006.SZ,2019-01-31,-0.563208,-0.691471,-0.239359,0.004437,0.159091


## 2. Out-of-Sample IC and Rank IC

The IC test uses the size-and-industry-neutralized CMM signal. Its month-by-month correlation with next-month returns should be stronger and more stable than equal-weighted momentum.

In [3]:
def monthly_ic(frame, factor_col, ret_col='target_1m_ret'):
    rows = []
    for date, g in frame.groupby('signal_date', sort=True):
        x = g[factor_col]
        y = g[ret_col]
        valid = x.notna() & y.notna()
        if valid.sum() < 50:
            continue
        rows.append({
            'signal_date': date,
            'factor': factor_col,
            'pearson_ic': x[valid].corr(y[valid], method='pearson'),
            'spearman_rank_ic': x[valid].corr(y[valid], method='spearman'),
            'n': int(valid.sum()),
        })
    return pd.DataFrame(rows)

ic = pd.concat([
    monthly_ic(df, 'cmm_neutral'),
    monthly_ic(df, 'same_window_mom_cs_z'),
], ignore_index=True)

def summarize_ic(ic_df):
    rows = []
    for factor, g in ic_df.groupby('factor'):
        for col in ['pearson_ic', 'spearman_rank_ic']:
            s = g[col].dropna()
            rows.append({
                'factor': factor,
                'metric': col,
                'months': len(s),
                'mean': s.mean(),
                'std': s.std(ddof=1),
                't_stat': s.mean() / (s.std(ddof=1) / np.sqrt(len(s))),
                'positive_rate': (s > 0).mean(),
            })
    return pd.DataFrame(rows)

ic_summary = summarize_ic(ic)
ic.to_csv(OUT_DIR / 'monthly_ic.csv', index=False)
ic_summary.to_csv(OUT_DIR / 'ic_summary.csv', index=False)
ic_summary


,factor,metric,months,mean,std,t_stat,positive_rate
0,cmm_neutral,pearson_ic,88,0.060088,0.085784,6.570900,0.772727
1,cmm_neutral,spearman_rank_ic,88,0.079023,0.096750,7.662043,0.829545
2,same_window_mom_cs_z,pearson_ic,88,0.000984,0.132379,0.069716,0.477273
3,same_window_mom_cs_z,spearman_rank_ic,88,-0.014904,0.152445,-0.917148,0.465909


In [4]:
fig, ax = plt.subplots(figsize=(10, 4.8))
for factor, label in [('cmm_neutral', 'CMM Neutralized'), ('same_window_mom_cs_z', 'Equal-weight momentum')]:
    g = ic[(ic['factor'] == factor)].sort_values('signal_date')
    ax.plot(g['signal_date'], g['spearman_rank_ic'].rolling(6, min_periods=1).mean(), linewidth=2, label=label)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('6-Month Rolling Rank IC')
ax.set_xlabel('Signal date')
ax.set_ylabel('Rank IC')
ax.legend()
fig.tight_layout()
ic_path = OUT_DIR / 'rolling_rank_ic.png'
fig.savefig(ic_path, dpi=160, bbox_inches='tight')
plt.close(fig)
ic_path


PosixPath('../result/reports/cmm_explain/rolling_rank_ic.png')

## 3. Incremental Information Beyond Equal-Weighted Momentum

This is the most direct diagnostic. Each month, run a cross-sectional regression:

```text
next_month_return = a + b1 * CMM + b2 * equal_weight_momentum + error
```

If `b1` stays positive and significant while `b2` is controlled for, CMM is adding information beyond ordinary momentum.

In [5]:
def monthly_cs_regression(frame, y_col, x_cols):
    rows = []
    for date, g in frame.groupby('signal_date', sort=True):
        use = g[[y_col] + x_cols].replace([np.inf, -np.inf], np.nan).dropna()
        if len(use) < len(x_cols) + 30:
            continue
        y = use[y_col].to_numpy(dtype=float)
        x = use[x_cols].to_numpy(dtype=float)
        x = np.column_stack([np.ones(len(use)), x])
        beta, *_ = np.linalg.lstsq(x, y, rcond=None)
        row = {'signal_date': date, 'n': len(use), 'intercept': beta[0]}
        for name, value in zip(x_cols, beta[1:]):
            row[name] = value
        rows.append(row)
    return pd.DataFrame(rows)

reg = monthly_cs_regression(df, 'target_1m_ret_cs_z', ['cmm_neutral', 'same_window_mom_cs_z'])
reg_summary = []
for col in ['cmm_neutral', 'same_window_mom_cs_z']:
    s = reg[col].dropna()
    reg_summary.append({
        'coefficient': col,
        'months': len(s),
        'mean_slope': s.mean(),
        't_stat': s.mean() / (s.std(ddof=1) / np.sqrt(len(s))),
        'positive_rate': (s > 0).mean(),
    })
reg_summary = pd.DataFrame(reg_summary)
reg.to_csv(OUT_DIR / 'monthly_incremental_regression.csv', index=False)
reg_summary.to_csv(OUT_DIR / 'incremental_regression_summary.csv', index=False)
reg_summary


,coefficient,months,mean_slope,t_stat,positive_rate
0,cmm_neutral,88,0.065975,6.885065,0.795455
1,same_window_mom_cs_z,88,0.004936,0.347073,0.534091


## 4. Is CMM Just A Rescaled Momentum Signal?

If CMM were merely standard momentum in disguise, the cross-sectional correlation between CMM and equal-weighted momentum would be close to 1 and CMM would not add much after controlling for momentum.

In [6]:
corr_rows = []
for date, g in df.groupby('signal_date', sort=True):
    corr_rows.append({
        'signal_date': date,
        'cmm_neutral_vs_same_window_mom_rank_corr': g['cmm_neutral'].corr(g['same_window_mom_cs_z'], method='spearman'),
        'cmm_neutral_vs_same_window_mom_pearson_corr': g['cmm_neutral'].corr(g['same_window_mom_cs_z'], method='pearson'),
    })
factor_corr = pd.DataFrame(corr_rows)
factor_corr_summary = factor_corr.drop(columns='signal_date').agg(['mean','std','min','max']).T
factor_corr.to_csv(OUT_DIR / 'monthly_factor_correlations.csv', index=False)
factor_corr_summary.to_csv(OUT_DIR / 'factor_correlation_summary.csv')
factor_corr_summary


,mean,std,min,max
cmm_neutral_vs_same_window_mom_rank_corr,0.122796,0.231684,-0.325414,0.472019
cmm_neutral_vs_same_window_mom_pearson_corr,-0.005282,0.190214,-0.378951,0.314036


## 5. What Did The Network Learn To Weight?

Given the paper's mechanism, the learned scalar `z_hat` controls how concentrated the softmax weights are and whether positive or negative historical-return days receive more emphasis. Here we reconstruct the 231-day weights for the test set.

In [7]:
ret_mat = df[return_cols].to_numpy(dtype=np.float32)
z_hat = df['z_hat'].to_numpy(dtype=np.float32).reshape(-1, 1)
scores = z_hat * ret_mat
scores = scores - scores.max(axis=1, keepdims=True)
w = np.exp(scores).astype(np.float32)
w = w / w.sum(axis=1, keepdims=True)

ret_mean = ret_mat.mean(axis=1)
weighted_ret = (w * ret_mat).sum(axis=1)
positive_weight_share = (w * (ret_mat > 0)).sum(axis=1)
negative_weight_share = (w * (ret_mat < 0)).sum(axis=1)
effective_days = 1.0 / np.square(w).sum(axis=1)

df['equal_weight_window_mean'] = ret_mean
df['learned_weight_window_return'] = weighted_ret
df['weighting_gain_vs_equal_mean'] = weighted_ret - ret_mean
df['positive_weight_share'] = positive_weight_share
df['negative_weight_share'] = negative_weight_share
df['effective_weighted_days'] = effective_days

weight_summary = df[['z_hat','max_weight','positive_weight_share','negative_weight_share','effective_weighted_days','weighting_gain_vs_equal_mean']].describe(percentiles=[.01,.05,.25,.5,.75,.95,.99]).T
weight_summary.to_csv(OUT_DIR / 'weight_mechanism_summary.csv')
weight_summary


,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
z_hat,367699.0,0.152491,2.673422,-20.901148,-7.234758,-4.313279,-1.193623,0.102620,1.590385,4.712302,6.671958,13.569743
max_weight,367699.0,0.005445,0.001799,0.004329,0.004337,0.004372,0.004576,0.004966,0.005745,0.007933,0.011393,0.465667
positive_weight_share,367699.0,0.477511,0.044384,0.000000,0.363160,0.405701,0.450405,0.478320,0.505920,0.548443,0.578895,0.667143
negative_weight_share,367699.0,0.491523,0.045370,0.000000,0.385470,0.418394,0.462267,0.491947,0.521209,0.564503,0.597684,0.773965
effective_weighted_days,367699.0,229.502747,3.666117,4.583270,214.704572,224.428050,229.573715,230.668610,230.945541,230.998291,230.999924,231.001022
weighting_gain_vs_equal_mean,367699.0,-0.000207,0.002958,-0.495480,-0.010074,-0.005091,-0.000984,0.000061,0.001045,0.003465,0.006205,0.163014


In [8]:
# Average weight profile over event time t-252 ... t-22 for high/low z_hat groups.
df['z_hat_decile'] = df.groupby('signal_date')['z_hat'].transform(lambda s: pd.qcut(s.rank(method='first'), 10, labels=False) + 1)
low_idx = df['z_hat_decile'].eq(1).to_numpy()
high_idx = df['z_hat_decile'].eq(10).to_numpy()
profile = pd.DataFrame({
    'lag': [int(c.replace('ret_lag_', '')) for c in return_cols],
    'avg_weight_low_z_hat': w[low_idx].mean(axis=0),
    'avg_weight_high_z_hat': w[high_idx].mean(axis=0),
})
profile.to_csv(OUT_DIR / 'weight_profile_by_z_hat_decile.csv', index=False)

fig, ax = plt.subplots(figsize=(10, 4.8))
ax.plot(profile['lag'], profile['avg_weight_low_z_hat'], label='Low z_hat decile', linewidth=2)
ax.plot(profile['lag'], profile['avg_weight_high_z_hat'], label='High z_hat decile', linewidth=2)
ax.invert_xaxis()
ax.set_title('Average Learned Daily Weights by z_hat Decile')
ax.set_xlabel('Return lag, trading days before signal')
ax.set_ylabel('Average softmax weight')
ax.legend()
fig.tight_layout()
weight_profile_path = OUT_DIR / 'weight_profile_by_z_hat_decile.png'
fig.savefig(weight_profile_path, dpi=160, bbox_inches='tight')
plt.close(fig)
weight_profile_path


PosixPath('../result/reports/cmm_explain/weight_profile_by_z_hat_decile.png')

## 6. Which Characteristics Move `z_hat`?

The paper's interpretation is characteristic-managed momentum: the same past return path can be weighted differently for different firms. This section checks which firm characteristics are most associated with the learned scalar.

In [9]:
feature_corr_rows = []
for col in feature_cols:
    vals = []
    for date, g in df.groupby('signal_date', sort=True):
        c = g[col].corr(g['z_hat'], method='spearman')
        if pd.notna(c):
            vals.append(c)
    if vals:
        s = pd.Series(vals)
        feature_corr_rows.append({
            'feature': col,
            'mean_rank_corr_with_z_hat': s.mean(),
            'abs_mean_rank_corr': abs(s.mean()),
            't_stat': s.mean() / (s.std(ddof=1) / np.sqrt(len(s))) if s.std(ddof=1) > 0 else np.nan,
            'positive_rate': (s > 0).mean(),
        })
feature_corr = pd.DataFrame(feature_corr_rows).sort_values('abs_mean_rank_corr', ascending=False)
feature_corr.to_csv(OUT_DIR / 'feature_correlations_with_z_hat.csv', index=False)
feature_corr.head(20)


,feature,mean_rank_corr_with_z_hat,abs_mean_rank_corr,t_stat,positive_rate
0,z_amount,-0.744269,0.744269,-124.991664,0.000000
14,z_pv_amount_mean_1m,-0.734685,0.734685,-65.479670,0.000000
15,z_pv_amount_mean_3m,-0.693913,0.693913,-65.963690,0.000000
16,z_pv_amount_mean_6m,-0.659362,0.659362,-68.818991,0.000000
17,z_pv_amount_mean_12m,-0.606863,0.606863,-64.752183,0.000000
2,z_nmv,-0.554842,0.554842,-63.589657,0.000000
1,z_volume,-0.548161,0.548161,-68.662719,0.000000
6,z_pv_mom_12m,-0.501174,0.501174,-43.461946,0.000000
5,z_pv_mom_6m,-0.385329,0.385329,-27.956454,0.000000
7,z_pv_vol_1m,-0.375608,0.375608,-19.181565,0.045455


In [10]:
top = feature_corr.head(15).sort_values('mean_rank_corr_with_z_hat')
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(top['feature'], top['mean_rank_corr_with_z_hat'])
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Top Characteristics Associated With z_hat')
ax.set_xlabel('Mean monthly Spearman correlation')
fig.tight_layout()
feature_path = OUT_DIR / 'top_features_with_z_hat.png'
fig.savefig(feature_path, dpi=160, bbox_inches='tight')
plt.close(fig)
feature_path


PosixPath('../result/reports/cmm_explain/top_features_with_z_hat.png')

## 7. Summary Table

This table condenses the explanation into a few diagnostics: prediction strength, incremental information, relationship with equal-weighted momentum, and learned-weight behavior.

In [11]:
summary_rows = []
rank_ic_summary = ic_summary[ic_summary['metric'] == 'spearman_rank_ic'].set_index('factor')
summary_rows.append({'diagnostic':'CMM Neutralized mean rank IC', 'value': rank_ic_summary.loc['cmm_neutral','mean']})
summary_rows.append({'diagnostic':'Equal-weight momentum mean rank IC', 'value': rank_ic_summary.loc['same_window_mom_cs_z','mean']})
summary_rows.append({'diagnostic':'CMM Neutralized incremental regression slope t-stat', 'value': reg_summary.set_index('coefficient').loc['cmm_neutral','t_stat']})
summary_rows.append({'diagnostic':'Momentum incremental regression slope t-stat', 'value': reg_summary.set_index('coefficient').loc['same_window_mom_cs_z','t_stat']})
summary_rows.append({'diagnostic':'Mean rank corr: CMM Neutralized vs equal-weight momentum', 'value': factor_corr['cmm_neutral_vs_same_window_mom_rank_corr'].mean()})
summary_rows.append({'diagnostic':'Median effective weighted days', 'value': df['effective_weighted_days'].median()})
summary_rows.append({'diagnostic':'Median max daily weight', 'value': df['max_weight'].median()})
summary_rows.append({'diagnostic':'Mean positive-day weight share', 'value': df['positive_weight_share'].mean()})
summary = pd.DataFrame(summary_rows)
summary.to_csv(OUT_DIR / 'explanation_summary.csv', index=False)
summary


,diagnostic,value
0,CMM Neutralized mean rank IC,0.079023
1,Equal-weight momentum mean rank IC,-0.014904
2,CMM Neutralized incremental regression slope t...,6.885065
3,Momentum incremental regression slope t-stat,0.347073
4,Mean rank corr: CMM Neutralized vs equal-weigh...,0.122796
5,Median effective weighted days,230.668610
6,Median max daily weight,0.004966
7,Mean positive-day weight share,0.477511
